# Run RWE on Reddit Politosphere (behavioral ideological axis)

Unlike the MIND notebook (where the left–right axis is a *text* proxy scored from
headlines — a weak, topic-vs-stance-conflated signal), this runs RWE on the
**Reddit Politosphere** and learns the ideological axis from **behavior**: who
participates in which political subreddit. That's the ideal-point method
(`rwe.IdeologyModel`) — a *validated* ideology measure — so RQ3's bridging is on a
**genuine** axis, with a clean `lean_corr` validation number (no Twitter API).

Pipeline: download a slice → ingest to a user×subreddit `.npz` → fit the ideal-point
axis (oriented to known subreddit leans) → RQ2/RQ3 + the axis plot. Everything is
cached to Drive, so it survives runtime resets.

> **License:** confirm the terms on <https://zenodo.org/records/5851729> before use.
> Politosphere is pseudonymized and derived from Pushshift; nothing is committed.

In [ ]:
# 1) Get the code (branch with the Politosphere pipeline) and install it
import os
if not os.path.isdir("/content/random_walks_with_erasure"):
    get_ipython().system("git clone --branch claude/sleepy-gates-oecof1 https://github.com/greenwichg/random_walks_with_erasure.git /content/random_walks_with_erasure")
else:
    get_ipython().system("git -C /content/random_walks_with_erasure pull -q")   # pick up fixes
os.chdir("/content/random_walks_with_erasure")
get_ipython().system("pip install -e . -q")
print("installed ->", os.getcwd())

In [ ]:
# 1b) Drive cache — make every expensive artifact (the comment files, the .npz)
#     survive Colab runtime resets. Mounts Drive once; later cells call
#     cache_get / cache_put, so after the first run you never re-download.
import os, shutil

CACHE = "/content/drive/MyDrive/rwe_polito"
try:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(CACHE, exist_ok=True)
    CACHE_OK = True
    print("Drive cache ready ->", CACHE)
except Exception as e:
    CACHE_OK = False
    print("(no Drive cache; artifacts will NOT persist across resets):", e)

def cache_get(name):
    """Copy <name> back from the Drive cache into the working dir if available."""
    src = os.path.join(CACHE, os.path.basename(name))
    if CACHE_OK and os.path.exists(src) and not os.path.exists(name):
        os.makedirs(os.path.dirname(name) or ".", exist_ok=True)
        shutil.copy(src, name)
        print("restored from Drive cache:", name)
    return os.path.exists(name)

def cache_put(name):
    """Save <name> to the Drive cache for future runs."""
    if CACHE_OK and os.path.exists(name):
        shutil.copy(name, os.path.join(CACHE, os.path.basename(name)))
        print("cached to Drive:", name)

In [ ]:
# 2) Download a slice of Politosphere from Zenodo (record 5851729). Default: the
#    US-2016-election window (matches the original RWE paper's 'US elections 2016').
#    Cached to Drive -> reset-safe. Add/extend MONTHS for more data (bigger = slower).
import os, glob
os.makedirs("politosphere", exist_ok=True)
MONTHS = ["2016-09", "2016-10", "2016-11"]
BASE = "https://zenodo.org/records/5851729/files"
for m in MONTHS:
    path = f"politosphere/comments_{m}.bz2"
    if cache_get(path):
        continue
    print("downloading", os.path.basename(path), "...")
    get_ipython().system(f"wget -q -O {path} '{BASE}/comments_{m}.bz2?download=1'")
    ok = (os.path.exists(path) and os.path.getsize(path) > 10000
          and open(path, "rb").read(3) == b"BZh")          # real bzip2, not an HTML 404
    if ok:
        cache_put(path)
    else:
        if os.path.exists(path):
            os.remove(path)
        print(f"  !! comments_{m}.bz2 did not download as a .bz2. Check the exact file "
              "name in the Files section of https://zenodo.org/records/5851729 (it may "
              "be bundled differently), or download it manually and drop it into the "
              "politosphere/ folder, then re-run.")
print("have:", sorted(glob.glob("politosphere/*.bz2")))

In [ ]:
# 3) Ingest -> user×subreddit matrix + ideal-point ideology axis (oriented to the
#    bundled subreddit leans). Cached to Drive (reset-safe).
#    NOTE: at the default --min-item-clicks 20 the axis comes out NULL
#    (lean_corr ~0.13, scrambled extremes) -- niche/small subreddits add idiosyncratic
#    noise that swamps left-right. The VALIDATED run is cell 3b
#    (--min-item-clicks 200 -> lean_corr ~0.65). This cell is kept to show the
#    threshold-sensitivity; report cell 3b.
if not cache_get("politosphere.npz"):
    get_ipython().system("python examples/ingest_politosphere.py --comments-dir politosphere "
                         "--ideology --min-user-clicks 5 --min-item-clicks 20 "
                         "--sample-users 15000 --out politosphere.npz")
    cache_put("politosphere.npz")
# WATCH the printed lean_corr: at min-item 20 it is ~0.13 (the niche-noise null);
# cell 3b filters the low-signal subs and recovers the clean axis (~0.65).

In [ ]:
# 3b) THE VALIDATED CONFIG. A much higher --min-item-clicks drops the niche/small
#      subreddits (the low-signal noise that scrambles the axis at the default 20) and
#      keeps the big partisan ones. This is the run that VALIDATES the axis:
#      lean_corr ~0.65 (vs ~0.13 at min-item 20), with cleanly L/R-sorted extremes.
#      Separate .npz, so cell 3's (null) result is untouched. Fast (int-coded ingest).
MIN_ITEM = 200            # 200 validates; push 300/500 to keep only the largest subs
NPZ = f"politosphere_mi{MIN_ITEM}.npz"
if not cache_get(NPZ):
    get_ipython().system(f"python examples/ingest_politosphere.py --comments-dir politosphere "
                         f"--ideology --min-user-clicks 5 --min-item-clicks {MIN_ITEM} "
                         f"--sample-users 15000 --out {NPZ}")
    cache_put(NPZ)
# WATCH lean_corr (~0.65 = validated) and read the extremes -- communism/anarchism on
# the left, Trump/Farage/conservatives on the right.
get_ipython().system(f"python examples/eval_mind.py --npz {NPZ} --no-bprmf")
from rwe.mind import MINDData
import numpy as np
d = MINDData.load(NPZ); o = np.argsort(d.item_positions); ids = np.asarray(d.dataset.item_ids)
print("\nitems kept:", d.n_items)
print("LEFT-most :", [f"r/{s}({p:+.1f})" for s, p in zip(ids[o[:15]],  d.item_positions[o[:15]])])
print("RIGHT-most:", [f"r/{s}({p:+.1f})" for s, p in zip(ids[o[-15:]], d.item_positions[o[-15:]])])

In [ ]:
# 4) Evaluate on the VALIDATED axis (cell 3b's NPZ): baselines + RWE-D/RWE-B ->
#    RQ2 (long-tail) + RQ3 (ideological bridging). Same driver as MIND -- drop-in.
#    Writes the CSV that gets folded into RESULTS.md / the paper.
get_ipython().system(f"python examples/eval_mind.py --npz {NPZ} --no-bprmf "
                     f"--out-csv politosphere_results.csv")
import pandas as pd
print(f"\nRESULTS (Politosphere, validated ideal-point axis, {NPZ}):")
print(pd.read_csv("politosphere_results.csv", index_col=0).round(3).to_string())

In [ ]:
# 5) Where users + subreddits sit on the learned (validated) left<->right axis
get_ipython().system(f"python examples/plot_axis.py --npz {NPZ} --out polito_axis.png")
from IPython.display import Image, display
display(Image("polito_axis.png"))
try:
    from google.colab import files; files.download("polito_axis.png")
except Exception:
    pass

## What to look at

- **`lean_corr`** — the headline number. At the default filter (cell 3,
  `--min-item-clicks 20`) it is ~0.13 (niche-subreddit noise). **Filter the
  low-signal subs (cell 3b, `--min-item-clicks 200`) and it jumps to ~0.65** — a
  *clean* ideological axis, vs the MIND headline proxy's ~0. That validated run is
  the one to report.
- **The extremes** (cell 3b) — LEFT = communism/anarchism/socialism, RIGHT =
  Trump/Farage/conservatives. An unmistakable left–right ordering from behavior alone.
- **RQ3 table** (cells 3b/4) — `uw_shift` / `uw_recs` for RWE-B vs the baselines, now
  on a *validated* ideological axis. RWE-B bridges hardest (`uw_shift` ~1.93).
- **The axis plot** (cell 5) — left-leaning subreddits on the left, right on the right,
  users spread between them.
- **The health report** (cell 6) — the *inverse* of the MIND report: Topic / Reporting /
  Emotion go n/a, but **Viewpoint Balance + Echo Chamber sit on the validated axis**
  (the metrics MIND couldn't support), and Source Diversity = community breadth.
  `--domain reddit` relabels the nouns.

This is folded into `RESULTS.md` / the paper as a third dataset — the one that gives
RQ3 a **validated** axis (with honest caveats: n=20 labels, single seed,
threshold-sensitive).

**Scale:** start with a few months; the de-dup grows with #users, and the ideal-point
fit is dense `O(users×items)` (capped by `--sample-users`). Extend `MONTHS` once a
small run works end-to-end.

## What to look at

- **`lean_corr`** — the headline number. At the default filter (cell 3,
  `--min-item-clicks 20`) it is ~0.13 (niche-subreddit noise). **Filter the
  low-signal subs (cell 3b, `--min-item-clicks 200`) and it jumps to ~0.65** — a
  *clean* ideological axis, vs the MIND headline proxy's ~0. That validated run is
  the one to report.
- **The extremes** (cell 3b) — LEFT = communism/anarchism/socialism, RIGHT =
  Trump/Farage/conservatives. An unmistakable left–right ordering from behavior alone.
- **RQ3 table** (cells 3b/4) — `uw_shift` / `uw_recs` for RWE-B vs the baselines, now
  on a *validated* ideological axis. RWE-B bridges hardest (`uw_shift` ~1.93).
- **The axis plot** (cell 5) — left-leaning subreddits on the left, right on the right,
  users spread between them.

This is folded into `RESULTS.md` / the paper as a third dataset — the one that gives
RQ3 a **validated** axis (with honest caveats: n=20 labels, single seed,
threshold-sensitive).

**Scale:** start with a few months; the de-dup grows with #users, and the ideal-point
fit is dense `O(users×items)` (capped by `--sample-users`). Extend `MONTHS` once a
small run works end-to-end.